## Exercise — Loan-Approval Audit (XAI Memo on 2 Flagged Cases)

You are the AI Risk Officer at UdaciBank. The new loan-approval model has generated consumer complaints from declined applicants. Two cases are on your desk for an audit memo that the model-risk + legal teams will rely on.

**Data:**
- `data/loan_classifier.joblib` — the production gradient-boosting model
- `data/loan_test.csv` — the held-out test set
- `data/flagged_cases.csv` — the two flagged declined applicants

**Deliverable (this notebook):**
1. SHAP per-case waterfall plots saved as `shap_explanations_case1.png` + `shap_explanations_case2.png`.
2. Global SHAP feature-importance bar chart saved as `shap_global_importance.png`.
3. A `detect_proxy_features()` helper that flags features acting as proxies for protected attributes.
4. The audit memo embedded as a markdown cell at the end of the notebook.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

bundle = joblib.load("data/loan_classifier.joblib")
clf = bundle["model"]
feature_names = bundle["feature_names"]

test    = pd.read_csv("data/loan_test.csv")
flagged = pd.read_csv("data/flagged_cases.csv")
print(f"Loaded model + {len(feature_names)} features. Test rows = {len(test)}; Flagged cases = {len(flagged)}")
flagged

## 1. SHAP for the two flagged cases

In [ ]:
# TODO: Build a SHAP explainer for the loan classifier, using the test set as the
#   background distribution. Then compute SHAP values for (a) the full test set
#   and (b) the two flagged cases.
#
#   For each flagged case, print:
#     - the model's predicted approval probability
#     - the top 5 features by absolute SHAP value, with the sign of each contribution
#
#   Pick the SHAP explainer class that matches a tree-based sklearn model; check the
#   SHAP docs for guidance on what "background distribution" means for that class.
pass


## 2. Save per-case SHAP waterfall plots

In [ ]:
# TODO: For each flagged case, render a SHAP waterfall plot showing how each feature
#   contributed to the model's prediction, and save the plots as:
#     - shap_explanations_case1.png
#     - shap_explanations_case2.png
#
#   The SHAP docs cover the Explanation object and the waterfall plot API.
pass


## 3. Global feature importance

In [ ]:
# TODO Step 6: render and save shap_global_importance.png as a horizontal bar chart of
#   global mean |SHAP| per feature, sorted descending.
pass

## 4. Detect proxy features

In [ ]:
def detect_proxy_features(local_shap_matrix, full_shap_matrix, feature_names,
                          suspect_features, multiplier=2.0):
    """Flag features whose mean local-impact across the flagged cases is materially larger
    than the model's global mean impact.

    Args:
        local_shap_matrix: SHAP values for ONLY the flagged cases (shape: n_flagged × n_features).
        full_shap_matrix:  SHAP values for the FULL test set (shape: n_test × n_features) —
                           the global baseline.
        feature_names:     list of feature names matching matrix columns.
        suspect_features:  list of feature names that are candidate proxies (e.g.,
                           zip_income_index, employer_size_score, education_score).
        multiplier:        how many times larger local impact must be vs the global baseline
                           before a feature is flagged. Default 2.0 means local impact must
                           be at least 2× global before the feature is flagged.

    Returns:
        DataFrame with one row per feature, columns: feature, local_mean, global_mean,
        ratio, flagged_as_proxy.
    """
    # TODO: For each feature, compute a per-feature comparison between local impact
    #   (across the flagged cases) and global impact (across the full test set).
    #   Return a DataFrame with one row per feature including the local mean, global
    #   mean, ratio, and a boolean flag.
    #
    #   A feature should be flagged only if it appears in suspect_features AND its
    #   local impact materially exceeds the global baseline (the multiplier governs
    #   how strict "materially" is).
    pass


SUSPECT = ["zip_income_index", "employer_size_score", "education_score"]
# proxy_report = detect_proxy_features(shap_values_flagged, shap_values_test, feature_names, SUSPECT)
# print(proxy_report)


## 5. Feature-removal counterfactual (the analytical step the demo doesn't cover)

The demo computed SHAP attribution only. Here you'll go one step further: re-score the two flagged cases with the most-suspect proxy feature ablated (set to a neutral value), and quantify how much approval probability and SHAP attribution change. This tells the AI Risk Officer whether the proxy was *actually* driving the decline or whether it was correlated with another driver.

In [ ]:
# TODO: From your proxy_report, identify the most-suspect feature (the one with
#   the highest ratio among the candidates flagged as proxies).

# TODO: Re-score the two flagged cases with that feature ablated to a neutral
#   baseline. A defensible neutral baseline is the test-set median — but you
#   could argue for the mean, mode, or domain-specific anchor instead; pick one
#   and justify it briefly in the audit memo.

# TODO: Report:
#   - the change in P(approve) for each case (Δ before → after)
#   - whether the decision flipped (was DECLINE, now APPROVE)
#   - the new top-3 |SHAP| features under the ablation, vs before

pass


## 6. Audit Memo (fill in below)

> **Audit Memo — UdaciBank Loan-Approval Model — Cases CASE-001 and CASE-002.**
>
> **Model behavior summary:**  [Your response here — what did the model do on both cases?]
>
> **Explanation evidence:**
> - **CASE-001** — top features (sign): [Your response here — list the top 5 from your SHAP analysis]. See `shap_explanations_case1.png`.
> - **CASE-002** — top features (sign): [Your response here — list the top 5 from your SHAP analysis]. See `shap_explanations_case2.png`.
>
> **Counterfactual finding:** [Your response here — which proxy did you ablate, what happened to P(approve), did the decision flip?]
>
> **Proxy-attribute findings:** [Your response here — which features did `detect_proxy_features()` flag, and what protected attribute does each one proxy for?]
>
> **Caveats on the limits of SHAP** *(include this caveat sentence verbatim or paraphrased)*: SHAP shows correlation between features and model output, not causation; values are local and depend on the background distribution; class-imbalance can amplify low-frequency-feature attribution.
>
> **Recommended next steps per case** *(pick from the menu — overturn / re-review / escalate to fairness audit / decline-stand)*:
> - **CASE-001:** [pick from menu, with one-sentence rationale]
> - **CASE-002:** [pick from menu, with one-sentence rationale]
> - **Model-level:** [pick from menu — recommendation on the model retrain (pull suspect features, run fairness audit, escalate to AIRB), with one-sentence rationale]

## Reference Notes

A few specification details for the libraries and legal anchors referenced above.

- **Detoxify model variants.** The package ships three checkpoints: `original` (BERT-based, `bert-base-uncased`), `unbiased` (RoBERTa-based), and `multilingual` (XLM-R-based). References to "RoBERTa / XLM-R variants" point specifically to the `unbiased` and `multilingual` checkpoints.
- **ECOA / Regulation B (2026).** ECOA + Reg B remain the operative federal fair-lending anchor. The CFPB's April 2026 final rule on Regulation B narrowed federal disparate-impact liability, while disparate-treatment analysis (including via proxies such as ZIP-derived income) remains actionable. State fair-lending laws and the Fair Housing Act also continue to recognize disparate-impact theories.